# Practical 05 — Settings in a File, Bugs Caught by a Robot

SCSE3040 Machine Learning Operations · Bennett University · Session 2026–27

This practical continues the same delivery-time project from P01 → P02 → P03 → P04. P05 adds YAML configuration and automated pytest testing to the existing P04 package.

In [1]:
from pathlib import Path
import sys
import subprocess
import yaml
import pandas as pd

P05_DIR = Path.cwd()
PRACTICALS_DIR = P05_DIR.parent
P04_WORK = PRACTICALS_DIR / "P04-from-notebook-to-package" / "work"
DATA_PATH = PRACTICALS_DIR / "data" / "delivery_times.csv"
WORK = P05_DIR / "work"
WORK.mkdir(exist_ok=True)

if str(P04_WORK) not in sys.path:
    sys.path.insert(0, str(P04_WORK))

print("P05:", P05_DIR)
print("P04 package:", P04_WORK)
print("Shared dataset:", DATA_PATH)
print("P04 package exists:", (P04_WORK / "delivery").is_dir())
print("Dataset exists:", DATA_PATH.exists())
assert (P04_WORK / "delivery").is_dir()
assert DATA_PATH.exists()


P05: C:\MLOps\mlops-course\Practicals\P05-settings-and-testing
P04 package: C:\MLOps\mlops-course\Practicals\P04-from-notebook-to-package\work
Shared dataset: C:\MLOps\mlops-course\Practicals\data\delivery_times.csv
P04 package exists: True
Dataset exists: True


In [2]:
orders = pd.read_csv(DATA_PATH)
print("Dataset shape:", orders.shape)
print("Columns:", list(orders.columns))
assert orders.shape == (600, 5)


Dataset shape: (600, 5)
Columns: ['distance_km', 'prep_time_min', 'traffic_level', 'rain', 'delivery_min']


## T1 — Put training settings in YAML

In [3]:
config_text = """data:
  path: ../data/delivery_times.csv
  target: delivery_min
  features:
    - distance_km
    - prep_time_min
    - traffic_level
    - rain

split:
  test_size: 0.2
  random_state: 42

model:
  type: LinearRegression
"""
(WORK / "config.yaml").write_text(config_text, encoding="utf-8")
config = yaml.safe_load((WORK / "config.yaml").read_text(encoding="utf-8"))
print(config)


{'data': {'path': '../data/delivery_times.csv', 'target': 'delivery_min', 'features': ['distance_km', 'prep_time_min', 'traffic_level', 'rain']}, 'split': {'test_size': 0.2, 'random_state': 42}, 'model': {'type': 'LinearRegression'}}


In [4]:
from delivery.data import load_orders, split_orders
from delivery.model import train_model, evaluate, save_model

def train_from_config(config_path=WORK / "config.yaml"):
    with open(config_path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    df = load_orders(P05_DIR / cfg["data"]["path"])
    X_train, X_test, y_train, y_test = split_orders(
        df,
        test_size=cfg["split"]["test_size"]
    )
    model = train_model(X_train, y_train)
    mae = evaluate(model, X_test, y_test)
    model_path = WORK / "model.joblib"
    save_model(model, model_path)
    return {"model": model, "mae": mae, "train_rows": len(X_train), "test_rows": len(X_test), "model_path": model_path, "config": cfg}

result = train_from_config()
print("Train rows:", result["train_rows"])
print("Test rows:", result["test_rows"])
print("MAE:", round(result["mae"], 4))


Train rows: 480
Test rows: 120
MAE: 1.9247


In [5]:
config["split"]["test_size"] = 0.3
with open(WORK / "config.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(config, f, sort_keys=False)
result_30 = train_from_config()
print("Configured test_size:", config["split"]["test_size"])
print("Test rows with 30% split:", result_30["test_rows"])
assert result["test_rows"] == 120
assert result_30["test_rows"] == 180


Configured test_size: 0.3
Test rows with 30% split: 180


## T2 — Test the existing P04 speed function

In [6]:
test_speed_code = '''from delivery.features import average_speed_kmph


def test_average_speed_basic():
    assert average_speed_kmph(10, 30) == 20


def test_average_speed_another_case():
    assert average_speed_kmph(20, 60) == 20
'''
(WORK / "test_speed.py").write_text(test_speed_code, encoding="utf-8")


212

## Test the existing P04 order validation

In [7]:
test_orders_code = '''from delivery.validate import is_valid_order


def test_valid_order():
    order = {"distance_km": 10, "prep_time_min": 25, "traffic_level": 3, "rain": 0}
    assert is_valid_order(order) is True


def test_invalid_distance():
    order = {"distance_km": 0, "prep_time_min": 25, "traffic_level": 3, "rain": 0}
    assert is_valid_order(order) is False


def test_invalid_traffic():
    order = {"distance_km": 10, "prep_time_min": 25, "traffic_level": 4, "rain": 0}
    assert is_valid_order(order) is False
'''
(WORK / "test_orders.py").write_text(test_orders_code, encoding="utf-8")


508

## T3 — Test model behaviour

In [8]:
test_model_code = '''import numpy as np
import yaml

from pathlib import Path
from delivery.data import load_orders, split_orders
from delivery.model import train_model

WORK = Path(__file__).resolve().parent
P05_DIR = WORK.parent


def test_model_predictions_are_finite_and_positive():
    config = yaml.safe_load((WORK / "config.yaml").read_text(encoding="utf-8"))
    df = load_orders(P05_DIR / config["data"]["path"])
    X_train, X_test, y_train, y_test = split_orders(
        df, test_size=config["split"]["test_size"]
    )
    model = train_model(X_train, y_train)
    predictions = model.predict(X_test)
    assert len(predictions) == len(X_test)
    assert np.isfinite(predictions).all()
    assert (predictions > 0).all()
'''
(WORK / "test_model.py").write_text(test_model_code, encoding="utf-8")


713

In [9]:
conftest_code = '''import sys
from pathlib import Path

P05_DIR = Path(__file__).resolve().parent.parent
P04_WORK = P05_DIR.parent / "P04-from-notebook-to-package" / "work"

if str(P04_WORK) not in sys.path:
    sys.path.insert(0, str(P04_WORK))
'''
(WORK / "conftest.py").write_text(conftest_code, encoding="utf-8")


227

## Run the complete pytest suite

In [10]:
completed = subprocess.run(
    [sys.executable, "-m", "pytest", "work", "-v"],
    cwd=P05_DIR,
    capture_output=True,
    text=True,
)
print(completed.stdout)
if completed.stderr:
    print(completed.stderr)
assert completed.returncode == 0


============================= test session starts =============================
platform win32 -- Python 3.14.7, pytest-9.1.1, pluggy-1.6.0 -- C:\MLOps\mlops-course\.venv\Scripts\python.exe
cachedir: .pytest_cache
rootdir: C:\MLOps\mlops-course\Practicals\P05-settings-and-testing
plugins: anyio-4.15.1
collecting ... collected 6 items

work/test_model.py::test_model_predictions_are_finite_and_positive PASSED [ 16%]
work/test_orders.py::test_valid_order PASSED                             [ 33%]
work/test_orders.py::test_invalid_distance PASSED                        [ 50%]
work/test_orders.py::test_invalid_traffic PASSED                         [ 66%]
work/test_speed.py::test_average_speed_basic PASSED                      [ 83%]
work/test_speed.py::test_average_speed_another_case PASSED               [100%]

============================== 6 passed in 1.67s ==============================



## Break a test deliberately

In [11]:
speed_path = WORK / "test_speed.py"
original_test_speed = speed_path.read_text(encoding="utf-8")
broken_test_speed = original_test_speed.replace(
    "assert average_speed_kmph(10, 30) == 20",
    "assert average_speed_kmph(10, 30) == 21",
)
speed_path.write_text(broken_test_speed, encoding="utf-8")

broken = subprocess.run(
    [sys.executable, "-m", "pytest", "work", "-q"],
    cwd=P05_DIR,
    capture_output=True,
    text=True,
)
print(broken.stdout)
print("Exit code:", broken.returncode)
assert broken.returncode != 0

speed_path.write_text(original_test_speed, encoding="utf-8")
fixed = subprocess.run(
    [sys.executable, "-m", "pytest", "work", "-q"],
    cwd=P05_DIR,
    capture_output=True,
    text=True,
)
print(fixed.stdout)
assert fixed.returncode == 0


....F.                                                                   [100%]
================================== FAILURES ===================================
__________________________ test_average_speed_basic ___________________________

    def test_average_speed_basic():
>       assert average_speed_kmph(10, 30) == 21
E       assert 20.0 == 21
E        +  where 20.0 = average_speed_kmph(10, 30)

work\test_speed.py:5: AssertionError
=========================== short test summary info ===========================
FAILED work/test_speed.py::test_average_speed_basic - assert 20.0 == 21
1 failed, 5 passed in 1.95s

Exit code: 1
......                                                                   [100%]
6 passed in 1.62s



## Final verification

P05 extends the same P01–P04 project using the shared dataset, existing P04 package, YAML configuration, pytest tests, and GitHub Actions.